In [40]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.window import Window

In [41]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("OlympicsDevelopment")
    .getOrCreate()
)

spark

In [42]:
DATA_ROOT = Path("../data/test_output")

BATCH_1_DATE = "2026-06-20"
BATCH_2_DATE = "2026-06-21"

#### Imports and loaded tables

In [43]:

bronze_athletes = spark.read.parquet(f'{DATA_ROOT}/bronze/athlete_events')
bronze_noc = spark.read.parquet(f'{DATA_ROOT}/bronze/noc_regions')
dim_athlete = spark.read.parquet(f'{DATA_ROOT}/gold/dim_athlete')
dim_games = spark.read.parquet(f'{DATA_ROOT}/gold/dim_games')
dim_event = spark.read.parquet(f'{DATA_ROOT}/gold/dim_event')
dim_noc = spark.read.parquet(f'{DATA_ROOT}/gold/dim_noc')
fact_participation = spark.read.parquet(f'{DATA_ROOT}/gold/fact_participation')


### Batch comparison

##### Comparison of size of both batches

In [26]:
# loaded data from athlete_events
batch_athletes_summary = (
    bronze_athletes
    .groupBy("_batch_date")
    .agg(
        f.count("*").alias("number_rows")
    )
    .select(
        f.col('_batch_date').alias('odate'),
        f.col('number_rows')
    )
    .orderBy(
        f.col('odate')
    )
)

batch_athletes_summary.show()

+----------+-----------+
|     odate|number_rows|
+----------+-----------+
|2026-06-20|       4831|
|2026-06-21|      14118|
+----------+-----------+



In [27]:
# loaded data from athlete_events
batch_noc_summary = (
    bronze_noc
    .groupBy("_batch_date")
    .agg(
        f.count("*").alias("number_rows")
    )
    .select(
        f.col('_batch_date').alias('odate'),
        f.col('number_rows')
    )
    .orderBy(
        f.col('odate')
    )
)

batch_noc_summary.show()

+----------+-----------+
|     odate|number_rows|
+----------+-----------+
|2026-06-20|         10|
|2026-06-21|         16|
+----------+-----------+



##### New athlete NOCs in second batch

In [34]:
batch_1_athletes = bronze_athletes.filter(f.col('_batch_date') == f.lit(BATCH_1_DATE).cast("date"))
batch_2_athletes = bronze_athletes.filter(f.col('_batch_date') == f.lit(BATCH_2_DATE).cast("date"))

new_nocs = (
    batch_2_athletes.select(f.col('NOC').alias('noc_code')).distinct()
    .join(
        batch_1_athletes.select(f.col('NOC').alias('noc_code')).distinct(),
        on = 'noc_code',
        how = 'left_anti'
    )
)

new_nocs.show()

+--------+
|noc_code|
+--------+
|     NED|
|     KOR|
|     SWE|
|     SGP|
|     CHN|
|     JPN|
+--------+



##### Comparison volume of participation by NOC

In [39]:
batch_1_noc_counts = (
    batch_1_athletes
    .groupBy("NOC")
    .agg(
        f.count("*").alias('batch_1_rows')
    )
)

batch_2_noc_counts = (
    batch_2_athletes
    .groupBy("NOC")
    .agg(
        f.count("*").alias('batch_2_rows')
    )
)

noc_volume_comparision = (
    batch_1_noc_counts
    .join(
        batch_2_noc_counts,
        on = 'NOC',
        how = 'full'
    )
    .fillna(
        0, subset=['batch_2_rows', 'batch_1_rows']
    )
    .withColumn(
        "row_difference",
        f.col('batch_2_rows') - f.col('batch_1_rows')
    )
    .orderBy(
        f.asc("batch_1_rows")
    )
)

noc_volume_comparision.show(20,truncate=False)

+---+------------+------------+--------------+
|NOC|batch_1_rows|batch_2_rows|row_difference|
+---+------------+------------+--------------+
|CHN|0           |1057        |1057          |
|JPN|0           |1080        |1080          |
|KOR|0           |976         |976           |
|NED|0           |859         |859           |
|SGP|0           |149         |149           |
|SWE|0           |1080        |1080          |
|POR|306         |454         |148           |
|BRA|349         |646         |297           |
|ESP|399         |718         |319           |
|AUS|455         |819         |364           |
|GBR|522         |880         |358           |
|CAN|560         |1080        |520           |
|FRA|560         |1080        |520           |
|GER|560         |1080        |520           |
|ITA|560         |1080        |520           |
|USA|560         |1080        |520           |
+---+------------+------------+--------------+



### SCD 2

##### Complete history of dim_noc

In [44]:
dim_noc.orderBy('noc_code', 'valid_from').show(100, truncate=False)

+--------+--------------------+---------------------------+---------------------------------------------+----------------------------------------------------------------+----------+----------+----------+
|noc_code|noc_key             |region                     |notes                                        |record_hash                                                     |valid_from|valid_to  |is_current|
+--------+--------------------+---------------------------+---------------------------------------------+----------------------------------------------------------------+----------+----------+----------+
|AUS     |-7160739004281565515|Australia                  |NULL                                         |c1ef40ce0484c698eb4bd27fe56c1e7b68d74f9780ed674210d0e5013dae45e9|2026-06-20|9999-12-31|true      |
|BRA     |4631191174186531624 |Brazil                     |NULL                                         |07f62b021771d3cf67e2e1faf18769cc5e5c119ad7d4d1847a11e11d6d5a7ecb|2026-06-20|999

In [45]:
scd_status_summary = (
    dim_noc
    .groupBy(
        "is_current"
    )
    .count()
    .orderBy(
        f.desc("is_current")
    )
)

scd_status_summary.show()

+----------+-----+
|is_current|count|
+----------+-----+
|      true|   16|
|     false|    3|
+----------+-----+



##### Old and new values 

In [50]:
noc_history_window = (Window.partitionBy("noc_code").orderBy("valid_from"))

scd_changes = (
    dim_noc
    .withColumn(
        'previous_region',
        f.lag('region').over(
            noc_history_window
        )
    )
    .withColumn(
        'previous_valid_from',
        f.lag('valid_from').over(
            noc_history_window
        )
    )
    .withColumn(
        'previous_valid_to',
        f.lag('valid_to').over(
            noc_history_window
        )
    )
    .filter(
        f.col('previous_region').isNotNull()
    )
    .filter(
        f.col('previous_region') != f.col('region')
    )
    .select(
        f.col('noc_code'),
        f.col('previous_region'),
        f.col('region').alias('new_region'),
        f.col('previous_valid_from'),
        f.col('previous_valid_to'),
        f.col('valid_from').alias('new_valid_from'),
        f.col('valid_to').alias('new_valid_to'),
        f.col('is_current')
    )
    .orderBy(
        f.col('noc_code')
    )
)

scd_changes.show(truncate=False)

+--------+---------------+---------------------------+-------------------+-----------------+--------------+------------+----------+
|noc_code|previous_region|new_region                 |previous_valid_from|previous_valid_to|new_valid_from|new_valid_to|is_current|
+--------+---------------+---------------------------+-------------------+-----------------+--------------+------------+----------+
|GER     |Germany        |Federal Republic of Germany|2026-06-20         |2026-06-20       |2026-06-21    |9999-12-31  |true      |
|POR     |Portugal       |Portuguese Republic        |2026-06-20         |2026-06-20       |2026-06-21    |9999-12-31  |true      |
|USA     |USA            |United States of America   |2026-06-20         |2026-06-20       |2026-06-21    |9999-12-31  |true      |
+--------+---------------+---------------------------+-------------------+-----------------+--------------+------------+----------+



In [52]:
version_timeline = (
    dim_noc
    .filter(
        f.col("noc_code").isin("GER", "POR", "USA")
    )
    .select(
        "noc_code",
        "region",
        "valid_from",
        "valid_to",
        "is_current",
        "record_hash",
    )
    .orderBy(
        "noc_code",
        "valid_from",
    )
)

version_timeline.show(truncate = False)

+--------+---------------------------+----------+----------+----------+----------------------------------------------------------------+
|noc_code|region                     |valid_from|valid_to  |is_current|record_hash                                                     |
+--------+---------------------------+----------+----------+----------+----------------------------------------------------------------+
|GER     |Germany                    |2026-06-20|2026-06-20|false     |80db4ccdca106d37b920206331fcfe3e9e50a9e763d89b54ce3ad5ac8cf30f03|
|GER     |Federal Republic of Germany|2026-06-21|9999-12-31|true      |f1c74a498810c95cbb38441251d79bb8c68b0f0980f584d5553e59466a10bf7a|
|POR     |Portugal                   |2026-06-20|2026-06-20|false     |4c7de2c3da6dc0ae8f9d13b107718a913859359bd30166d077c867577953865d|
|POR     |Portuguese Republic        |2026-06-21|9999-12-31|true      |8747632b73d90613ccf4dd07277169573fa71bf6e25b5fdae99773b4b80e26e5|
|USA     |USA                        |202

##### Detect new inserts

In [55]:
noc_lifecycle = (
    dim_noc
    .groupBy(f.col('noc_code'))
    .agg(
        f.count('*').alias('nb_of_versions'),
        f.min('valid_from').alias('first_seen'),
        f.max('valid_from').alias('latest_version'),
        f.max(
            f.when(
                f.col('is_current'), f.col('region')
            )
        ).alias('current_region')
    )
)

batch_2_inserts = (
    noc_lifecycle
    .filter(
        f.col('nb_of_versions') == 1
    )
    .filter(
        f.col('first_seen') == f.lit(BATCH_2_DATE).cast('date')
    )
    .select(
        f.col('noc_code'),
        f.col('current_region'),
        f.col('first_seen')
    )
    .orderBy(
        f.col('noc_code')
    )
)

batch_2_inserts.show(truncate=False)

+--------+--------------+----------+
|noc_code|current_region|first_seen|
+--------+--------------+----------+
|CHN     |China         |2026-06-21|
|JPN     |Japan         |2026-06-21|
|KOR     |South Korea   |2026-06-21|
|NED     |Netherlands   |2026-06-21|
|SGP     |Singapore     |2026-06-21|
|SWE     |Sweden        |2026-06-21|
+--------+--------------+----------+



##### Inserts, updates and unchanged records

In [57]:
noc_change_classification = (
    noc_lifecycle
    .withColumn(
        "change_type",
        f.when(
            f.col("nb_of_versions") > 1,
            f.lit("UPDATE")
        )
        .when(
            f.col("first_seen") == f.lit(BATCH_2_DATE).cast("date"),
            f.lit("INSERT")
        )
        .otherwise(
            f.lit("UNCHANGED")
        ),
    )
    .orderBy(
        "change_type",
        "noc_code",
    )
)

noc_change_classification.show(100,truncate=False)

+--------+--------------+----------+--------------+---------------------------+-----------+
|noc_code|nb_of_versions|first_seen|latest_version|current_region             |change_type|
+--------+--------------+----------+--------------+---------------------------+-----------+
|CHN     |1             |2026-06-21|2026-06-21    |China                      |INSERT     |
|JPN     |1             |2026-06-21|2026-06-21    |Japan                      |INSERT     |
|KOR     |1             |2026-06-21|2026-06-21    |South Korea                |INSERT     |
|NED     |1             |2026-06-21|2026-06-21    |Netherlands                |INSERT     |
|SGP     |1             |2026-06-21|2026-06-21    |Singapore                  |INSERT     |
|SWE     |1             |2026-06-21|2026-06-21    |Sweden                     |INSERT     |
|AUS     |1             |2026-06-20|2026-06-20    |Australia                  |UNCHANGED  |
|BRA     |1             |2026-06-20|2026-06-20    |Brazil                     |U

In [58]:
noc_change_classification.groupBy('change_type').count().orderBy('change_type').show()

+-----------+-----+
|change_type|count|
+-----------+-----+
|     INSERT|    6|
|  UNCHANGED|    7|
|     UPDATE|    3|
+-----------+-----+



In [59]:
spark.stop()